In [3]:
import pandas as pd
import networkx as nx
import requests
import pubchempy as pcp
from xml.etree import ElementTree as ET
import time
import csv
from tqdm import tqdm

In [4]:
df_cg = pd.read_csv('../edges/chemgene.csv')
df_cd = pd.read_csv('../edges/chemdis.csv')
df_gd = pd.read_csv('../edges/genedis.csv')

df_c = pd.read_csv('../vocab/CTD_chemicals.csv')
df_d = pd.read_csv('../vocab/CTD_diseases.csv')
df_g = pd.read_csv('../vocab/CTD_genes.csv')

df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneID'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseName'])  
df_gd = df_gd.drop_duplicates(subset=['GeneID', 'DiseaseName'])

In [5]:
print(df_cg.info())
print(df_cd.info())
print(df_gd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 812241 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        812241 non-null  object 
 1   ChemicalID          812241 non-null  object 
 2   CasRN               557625 non-null  object 
 3   GeneSymbol          812240 non-null  object 
 4   GeneID              812241 non-null  int64  
 5   GeneForms           809771 non-null  object 
 6   Organism            812241 non-null  object 
 7   OrganismID          812241 non-null  float64
 8   Interaction         812241 non-null  object 
 9   InteractionActions  812241 non-null  object 
 10  PubMedIDs           812241 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 74.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 3467034 entries, 0 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [6]:
known = pd.read_csv('chemsmiles.csv')
known.head()

,meshID,cids,smiles
0,C070055,35823,C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl
1,C024713,12389,CCCCCCCCCCCCCC
2,D000077612,166548,CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...
3,C538809,131634859,B(=O)OB1OB2OB(OB(O2)O1)[O-].[Na+]
4,C000626479,7565,C1=CC=C(C=C1)OC2=CC=C(C=C2)Br


In [7]:
df_cg = df_cg[df_cg["ChemicalID"].isin(known["meshID"])]
df_cd = df_cd[df_cd["ChemicalID"].isin(known["meshID"])]

print(df_cg.info())
print(df_cd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 703832 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        703832 non-null  object 
 1   ChemicalID          703832 non-null  object 
 2   CasRN               551580 non-null  object 
 3   GeneSymbol          703831 non-null  object 
 4   GeneID              703832 non-null  int64  
 5   GeneForms           701695 non-null  object 
 6   Organism            703832 non-null  object 
 7   OrganismID          703832 non-null  float64
 8   Interaction         703832 non-null  object 
 9   InteractionActions  703832 non-null  object 
 10  PubMedIDs           703832 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 64.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 2911974 entries, 1 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [8]:
G = nx.Graph()


In [9]:
chemicals = (set(df_cg['ChemicalID'].unique()) | set(df_cd['ChemicalID'].unique())) & set(known['meshID'])
genes = set(df_cg['GeneID'].unique()) | set(df_gd['GeneID'].unique())
diseases = set(df_cd['DiseaseID'].unique()) | set(df_gd['DiseaseID'].unique())

print(len(chemicals))
print(len(genes))
diseases = [dis[5:] for dis in diseases]
print(len(diseases))
print(len(chemicals)+len(genes)+len(diseases))

13943
28151
7239
49333


In [48]:
for chem in chemicals:
    G.add_node(chem, node_type='chemical')
for gene in genes:
    G.add_node(gene, node_type='gene')
for disease in diseases:
    G.add_node(disease, node_type='disease')

for _, row in df_cg.iterrows():
    G.add_edge(row['ChemicalID'], row['GeneID'], edge_type='chem_gene')
for _, row in df_cd.iterrows():
    G.add_edge(row['ChemicalID'], row['DiseaseID'], edge_type='chem_disease')
for _, row in df_gd.iterrows():
    G.add_edge(row['GeneID'], row['DiseaseID'], edge_type='gene_disease')

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph built: 56572 nodes, 3650027 edges


In [10]:
# df_c = df_c[['ChemicalID', 'ChemicalName', 'PubChemCID']].drop_duplicates(subset=['ChemicalID']).set_index('ChemicalID')
# df_d = df_d[['DiseaseID', 'DiseaseName', 'DiseaseSemanticType']].drop_duplicates(subset=['DiseaseID']).set_index('DiseaseID')
# df_g = df_g[['GeneID', 'GeneSymbol', 'GeneName']].drop_duplicates(subset=['GeneID']).set_index('GeneID')


print(df_c.info())
print(df_g.info())
print(df_d.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179408 entries, 0 to 179407
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   ChemicalName       179408 non-null  object
 1   ChemicalID         179408 non-null  object
 2   CasRN              56623 non-null   object
 3   Definition         6190 non-null    object
 4   ParentIDs          179407 non-null  object
 5   TreeNumbers        179408 non-null  object
 6   ParentTreeNumbers  179407 non-null  object
 7   Synonyms           116886 non-null  object
dtypes: object(8)
memory usage: 11.0+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641489 entries, 0 to 641488
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   GeneSymbol   641488 non-null  object
 1   GeneName     639154 non-null  object
 2   GeneID       641489 non-null  int64 
 3   AltGeneIDs   48428 non-null   object
 4

In [11]:
df_g = df_g[df_g["GeneID"].isin(genes)]
df_g.info()


<class 'pandas.core.frame.DataFrame'>
Index: 28149 entries, 15907 to 641485
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   GeneSymbol   28148 non-null  object
 1   GeneName     28096 non-null  object
 2   GeneID       28149 non-null  int64 
 3   AltGeneIDs   20825 non-null  object
 4   Synonyms     26173 non-null  object
 5   BioGRIDIDs   20212 non-null  object
 6   PharmGKBIDs  21696 non-null  object
 7   UniProtIDs   20417 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.9+ MB


In [12]:
paths = pd.read_csv('CTD_pathways.csv')
paths.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2567 entries, 0 to 2566
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   PathwayName  2567 non-null   object
 1   PathwayID    2567 non-null   object
dtypes: object(2)
memory usage: 40.2+ KB


In [13]:
dpath = pd.read_csv('../vocab/dispath.csv')
cpath = pd.read_csv('../vocab/chempath.csv')
gpath = pd.read_csv('../vocab/genepath.csv')

dpath['DiseaseID'] = dpath['DiseaseID'].str[5:]

cpath = cpath.drop_duplicates(subset=['ChemicalID', 'PathwayID'])  
gpath = gpath.drop_duplicates(subset=['GeneID', 'PathwayID'])
dpath = dpath.drop_duplicates(subset=['DiseaseID', 'PathwayID'])

print(dpath.head())
print(diseases)

cpath = cpath[cpath["ChemicalID"].isin(chemicals)]
gpath = gpath[gpath["GeneID"].isin(genes)]
dpath = dpath[dpath["DiseaseID"].isin(diseases)]

print(dpath.info())
print(cpath.info())
print(gpath.info())

                                  DiseaseName DiseaseID  \
0  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
1  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
2  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
3  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
4  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   

                                         PathwayName            PathwayID  \
0                              Androgen biosynthesis   REACT:R-HSA-193048   
1  Fatty acid, triacylglycerol, and ketone body m...   REACT:R-HSA-535734   
2                        Fatty Acyl-CoA Biosynthesis    REACT:R-HSA-75105   
3                                 Metabolic pathways        KEGG:hsa01100   
4                                         Metabolism  REACT:R-HSA-1430728   

  InferenceGeneSymbol  
0             HSD17B3  
1             HSD17B3  
2             HSD17B3  
3             HSD17B3  
4             HSD17B3  
['D016757', 'D055728', 'D014190', '300

In [14]:
pathways = set(dpath['PathwayID'].unique()) | set(cpath['PathwayID'].unique())  | set(gpath['PathwayID'].unique())
len(pathways)

2363

In [42]:
# Group PathwayIDs by DiseaseID
dpaths = dpath.groupby("DiseaseID")["PathwayID"].apply(list).reset_index()
dpaths = dpaths.rename(columns={"PathwayID": "PathwayIDs"})
dpaths.set_index('DiseaseID',inplace=True)
print(dpaths)
# Convert to list of dictionaries
result_dp = dpaths.to_dict(orient="records")



                                                  PathwayIDs
DiseaseID                                                   
102200     [REACT:R-HSA-8937144, REACT:R-HSA-211859, REAC...
105500     [REACT:R-HSA-983712, KEGG:hsa04978, KEGG:hsa04...
106190     [REACT:R-HSA-1280218, KEGG:hsa04925, KEGG:hsa0...
108120     [KEGG:hsa04261, KEGG:hsa04260, KEGG:hsa05414, ...
108420     [REACT:R-HSA-1640170, REACT:R-HSA-1500620, REA...
...                                                      ...
D066087    [REACT:R-HSA-1296052, REACT:R-HSA-418457, KEGG...
D066088    [REACT:R-HSA-74160, REACT:R-HSA-983712, REACT:...
D066126    [REACT:R-HSA-166054, REACT:R-HSA-5617472, REAC...
D066253    [KEGG:hsa05412, REACT:R-HSA-380108, KEGG:hsa04...
D066263    [KEGG:hsa05014, REACT:R-HSA-2262752, REACT:R-H...

[5051 rows x 1 columns]


In [43]:
# Group PathwayIDs by DiseaseID
gpaths = gpath.groupby("GeneID")["PathwayID"].apply(list).reset_index()
gpaths = gpaths.rename(columns={"PathwayID": "PathwayIDs"})
gpaths.set_index('GeneID',inplace=True)

# Convert to list of dictionaries
result_gp = gpaths.to_dict(orient="records")

print(len(result_gp))

11458


In [97]:
# Group PathwayIDs by DiseaseID
cpaths = cpath.groupby("ChemicalID")["PathwayID"].apply(list).reset_index()
cpaths = cpaths.rename(columns={"PathwayID": "PathwayIDs"})
cpaths.set_index('ChemicalID',inplace=True)
print(cpaths)
# Convert to list of dictionaries
result_cp = cpaths.to_dict(orient="records")

print(len(result_cp))

                                                   PathwayIDs
ChemicalID                                                   
C000015     [REACT:R-HSA-382556, KEGG:hsa02010, KEGG:hsa04...
C000050     [REACT:R-HSA-1368108, REACT:R-HSA-156590, REAC...
C000081     [REACT:R-HSA-442660, REACT:R-HSA-112316, REACT...
C000121     [REACT:R-HSA-166054, REACT:R-HSA-450341, KEGG:...
C000154     [REACT:R-HSA-111447, REACT:R-HSA-111459, REACT...
...                                                       ...
D064753                  [KEGG:hsa05418, REACT:R-HSA-6785807]
D064791     [REACT:R-HSA-166054, KEGG:hsa04920, KEGG:hsa05...
D065146     [REACT:R-HSA-450341, KEGG:hsa05221, KEGG:hsa04...
D065366     [REACT:R-HSA-166054, REACT:R-HSA-450341, KEGG:...
D065819     [REACT:R-HSA-2426168, REACT:R-HSA-5423646, KEG...

[8529 rows x 1 columns]
8529


In [49]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
import numpy as np

# Create generator once (faster)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024
)

def get_ecfp_embedding(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


In [118]:
from sklearn.preprocessing import MultiLabelBinarizer

# suppose you have something like:
# chem_id | pathway
# C1      | p53
# C1      | MAPK
# C2      | Apoptosis

pths = set(paths['PathwayID'].unique())
mlb_path = MultiLabelBinarizer()
print((pths))
pathway_emb = mlb_path.fit_transform([pths])
pathway_emb.shape

{'REACT:R-HSA-5696399', 'REACT:R-HSA-444821', 'REACT:R-HSA-5617472', 'REACT:R-HSA-71288', 'REACT:R-HSA-997272', 'KEGG:hsa_M00400', 'REACT:R-HSA-1655829', 'REACT:R-HSA-5688354', 'KEGG:hsa_M00681', 'REACT:R-HSA-3371598', 'REACT:R-HSA-6802953', 'REACT:R-HSA-418889', 'REACT:R-HSA-535734', 'REACT:R-HSA-5358346', 'REACT:R-HSA-2559586', 'REACT:R-HSA-6806942', 'REACT:R-HSA-1482839', 'REACT:R-HSA-8937144', 'REACT:R-HSA-69895', 'REACT:R-HSA-727802', 'REACT:R-HSA-4793952', 'KEGG:hsa00440', 'REACT:R-HSA-3359454', 'KEGG:hsa_M00092', 'REACT:R-HSA-2173795', 'REACT:R-HSA-5696397', 'REACT:R-HSA-417973', 'KEGG:hsa_M00086', 'REACT:R-HSA-5654708', 'REACT:R-HSA-5218900', 'REACT:R-HSA-3371556', 'REACT:R-HSA-4641257', 'KEGG:hsa04961', 'KEGG:hsa_M00158', 'REACT:R-HSA-1474151', 'REACT:R-HSA-390650', 'KEGG:hsa04213', 'REACT:R-HSA-211945', 'REACT:R-HSA-389599', 'REACT:R-HSA-112310', 'KEGG:hsa_M00014', 'REACT:R-HSA-187015', 'REACT:R-HSA-5368598', 'REACT:R-HSA-209560', 'REACT:R-HSA-5467348', 'REACT:R-HSA-2474795',

(1, 2567)

In [119]:
disease_slim = (
    df_d
    .groupby("DiseaseID")["SlimMappings"]
    .apply(lambda x: sorted(set(
        i for s in x.dropna() for i in s.split("|")
    )))
)
# slims = []
# for i in range(len(disease_slim)):
#     slims.extend(disease_slim.iloc[i])
# print(len(slims))
# print(len(set(slims)))

mlb_slim = MultiLabelBinarizer()
disease_slim_emb = mlb_slim.fit_transform(disease_slim)

disease_slim_emb.shape

(13317, 38)

In [120]:
def get_chemical_embedding(chem_id, smiles):
    ecfp = get_ecfp_embedding(smiles)
    try:
        pw = pathway_emb[cpaths.index.get_loc(chem_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    return np.concatenate([ecfp, pw]).astype(np.float32)


In [121]:
def get_disease_embedding(disease_id):
    try:
        slim = disease_slim_emb[disease_slim.index.get_loc(disease_id)]
    except:
        slim = np.zeros(13317, dtype=np.float32)
    try:
        pw = pathway_emb[dpaths.index.get_loc(disease_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    
    return np.concatenate([slim, pw]).astype(np.float32)


In [122]:
def get_gene_embedding(gene_id):
    try:
        pw = pathway_emb[gpaths.index.get_loc(gene_id)]
    except:
        pw = np.zeros(2567, dtype=np.float32)
    return pw.astype(np.float32)

In [123]:
from tqdm import tqdm

for chem_id in tqdm(chemicals):
    G.nodes[chem_id]['x'] = get_chemical_embedding(chem_id,known.loc[known["meshID"] == chem_id, "smiles"].iloc[0])
for disease_id in tqdm(diseases):
    G.nodes[disease_id]['x'] = get_disease_embedding(disease_id)
for gene_id in tqdm(genes):
    G.nodes[gene_id]['x'] = get_gene_embedding(gene_id)

  9%|▉         | 1227/13943 [00:02<00:19, 657.17it/s][13:43:05] WARNING: not removing hydrogen atom without neighbors
[13:43:05] WARNING: not removing hydrogen atom without neighbors
 39%|███▊      | 5384/13943 [00:08<00:13, 646.30it/s][13:43:12] WARNING: not removing hydrogen atom without neighbors
[13:43:12] WARNING: not removing hydrogen atom without neighbors
 64%|██████▍   | 8918/13943 [00:13<00:07, 696.47it/s][13:43:17] WARNING: not removing hydrogen atom without neighbors
[13:43:17] WARNING: not removing hydrogen atom without neighbors
 76%|███████▌  | 10586/13943 [00:18<00:06, 528.65it/s][13:43:21] WARNING: not removing hydrogen atom without neighbors
[13:43:21] WARNING: not removing hydrogen atom without neighbors
 79%|███████▉  | 11008/13943 [00:18<00:04, 605.57it/s][13:43:22] WARNING: not removing hydrogen atom without neighbors
[13:43:22] WARNING: not removing hydrogen atom without neighbors
[13:43:22] WARNING: not removing hydrogen atom without neighbors
[13:43:22] WARNING

In [124]:
import pickle
with open("bio_graph_raw.pkl", "wb") as f:
    pickle.dump(G, f)


In [125]:
import numpy as np

np.set_printoptions(linewidth=10**9)

with open("emb.txt", "w") as f:
    for g in genes:
        f.write(str(G.nodes[g]) + "\n")
